# 04 Dataset Builder

## Responsibility

02 Engineered Dataset에 03의 Selection 결과를 적용하여:

1. Model-ready Learning/Test Dataset 구성
2. Learning-only Prefix Evaluation Scenario 정의
3. 최종 Output 검증 및 저장

이 Notebook은 다음을 수행하지 않는다.
- Feature Transformation
- RUL Target Generation
- EDA
- Feature / Target Selection
- Scaling
- Sequence / Windowing

**Test 데이터는 03 EDA를 수행하지 않으며, 03에서 확정된 Selection 결과만 적용한다.**

In [ ]:
# =============================================================================
# 1. Imports & Configuration
# =============================================================================
from pathlib import Path
import json
import pandas as pd

from utils.dataset_utils import (
    ALLOWED_NAN_PREFIXES,
    DEFAULT_PREFIX_END,
    DEFAULT_PREFIX_START,
    DEFAULT_PREFIX_STEP,
    build_model_ready_dataset,
    create_prefix_definition,
    validate_prefix_definition,
)

print(
    "Allowed NaN Feature Prefixes : "
    f"{list(ALLOWED_NAN_PREFIXES)}"
)

INPUT_PATH = Path("outputs/02_feature_transformation/")
LEARNING_INPUT_PATH = INPUT_PATH / "learning_engineered_features.csv"
TEST_INPUT_PATH = INPUT_PATH / "test_engineered_features.csv"

SELECTION_MANIFEST_PATH = "outputs/03_eda_selection/selection_config.json"

OUTPUT_PATH = Path("outputs/04_dataset_builder/")
OUTPUT_PATH.mkdir(exist_ok=True)

LEARNING_OUTPUT_PATH = OUTPUT_PATH / "model_ready_learning_dataset.csv"
TEST_OUTPUT_PATH = OUTPUT_PATH / "model_ready_test_dataset.csv"
PREFIX_OUTPUT_PATH = OUTPUT_PATH / "prefix_definition.csv"
CONFIG_OUTPUT_PATH = OUTPUT_PATH / "dataset_config.json"

In [ ]:
# =============================================================================
# 2. Load Selection Manifest
# =============================================================================
with open(SELECTION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    selection_manifest = json.load(f)

if "candidate_features" not in selection_manifest:
    raise KeyError("Selection manifest must contain 'candidate_features'.")

if "selected_target" not in selection_manifest:
    raise KeyError("Selection manifest must contain 'selected_target'.")

CANDIDATE_FEATURES = selection_manifest["candidate_features"]
SELECTED_TARGET = selection_manifest["selected_target"]

if not isinstance(CANDIDATE_FEATURES, list):
    raise TypeError("'candidate_features' must be a list.")

if not isinstance(SELECTED_TARGET, str) or not SELECTED_TARGET:
    raise TypeError("'selected_target' must be a non-empty string.")

print(f"Selected Features : {len(CANDIDATE_FEATURES)}")
for feature in CANDIDATE_FEATURES:
    print(f"  - {feature}")
print(f"Selected Target   : {SELECTED_TARGET}")

In [ ]:
# =============================================================================
# 3. Load Engineered Datasets
# =============================================================================
learning_df = pd.read_csv(LEARNING_INPUT_PATH)
test_df = pd.read_csv(TEST_INPUT_PATH)

print("[Input Datasets]")
print(
    f"Learning : {learning_df.shape[0]:,} rows × "
    f"{learning_df.shape[1]:,} columns"
)
print(
    f"Test     : {test_df.shape[0]:,} rows × "
    f"{test_df.shape[1]:,} columns"
)

In [ ]:
# =============================================================================
# 4. Build Model-ready Datasets
# =============================================================================
print("=" * 60)
print("Building Model-ready Learning Dataset")
print("=" * 60)

model_learning_df = build_model_ready_dataset(
    learning_df,
    candidate_features=CANDIDATE_FEATURES,
    selected_target=SELECTED_TARGET,
)

print("=" * 60)
print("Building Model-ready Test Dataset")
print("=" * 60)

model_test_df = build_model_ready_dataset(
    test_df,
    candidate_features=CANDIDATE_FEATURES,
    selected_target=None,
)

print(
    f"Learning : {model_learning_df.shape[0]:,} rows × "
    f"{model_learning_df.shape[1]:,} columns"
)
print(
    f"Test     : {model_test_df.shape[0]:,} rows × "
    f"{model_test_df.shape[1]:,} columns"
)

In [ ]:
# =============================================================================
# 5. Model-ready Dataset Preview
# =============================================================================
print("[Model-ready Learning Dataset]")
display(model_learning_df.head(3))

print("[Model-ready Test Dataset]")
display(model_test_df.head(3))

In [ ]:
# =============================================================================
# 6. Prefix Definition — Learning Only
# =============================================================================
print("=" * 60)
print("Generating Prefix Evaluation Definition — Learning Only")
print("=" * 60)

prefix_definition = create_prefix_definition(
    model_learning_df,
    start=DEFAULT_PREFIX_START,
    end=DEFAULT_PREFIX_END,
    step=DEFAULT_PREFIX_STEP,
)

validate_prefix_definition(prefix_definition, model_learning_df)

print(f"Prefix rows : {len(prefix_definition):,}")
display(prefix_definition.head(10))

In [ ]:
# =============================================================================
# 7. Save Outputs
# =============================================================================
model_learning_df.to_csv(LEARNING_OUTPUT_PATH, index=False)
model_test_df.to_csv(TEST_OUTPUT_PATH, index=False)

prefix_definition.to_csv(PREFIX_OUTPUT_PATH, index=False)

dataset_config = {
    "candidate_features": CANDIDATE_FEATURES,
    "selected_target": SELECTED_TARGET,
    "nan_policy": {
        "allowed_feature_prefixes": list(ALLOWED_NAN_PREFIXES),
    },
    "prefix_settings": {
        "start": DEFAULT_PREFIX_START,
        "end": DEFAULT_PREFIX_END,
        "step": DEFAULT_PREFIX_STEP,
    },
}

with open(CONFIG_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_config, f, indent=4, ensure_ascii=False)

print("[Saved]")
print(f"Learning : {LEARNING_OUTPUT_PATH}")
print(f"Test     : {TEST_OUTPUT_PATH}")
print(f"Prefix   : {PREFIX_OUTPUT_PATH}")
print(f"Config   : {CONFIG_OUTPUT_PATH}")

In [ ]:
# =============================================================================
# 8. Final Summary
# =============================================================================
print("=" * 60)
print("04 Dataset Builder Complete")
print("=" * 60)
print(f"Selected Features : {len(CANDIDATE_FEATURES)}")
print(f"Selected Target   : {SELECTED_TARGET}")
print(
    f"Learning Dataset : {model_learning_df.shape[0]:,} rows × "
    f"{model_learning_df.shape[1]:,} columns"
)
print(
    f"Test Dataset     : {model_test_df.shape[0]:,} rows × "
    f"{model_test_df.shape[1]:,} columns"
)
print(f"Prefix Scenarios : {len(prefix_definition):,}")